# Brazilian E-Commerce (Olist) — ETL para Presentación Ejecutiva

Este notebook lee los datasets crudos de Olist desde `public/dataset/`, realiza el
análisis exploratorio y exporta archivos JSON agregados a `src/data/` para alimentar
los componentes de React de la presentación (Next.js + Reveal.js).

**Preguntas de negocio:**
1. ¿Cuál es el panorama general del negocio? (ingresos, órdenes, ticket, entregas, reseñas)
2. ¿Cómo impacta la latencia logística en la satisfacción del cliente?
3. ¿Cuánto dinero está en riesgo por la insatisfacción?
4. ¿Qué categorías concentran volumen, flete caro y malas reseñas?


In [1]:
import json
from pathlib import Path

import pandas as pd

DATASET = Path("public/dataset")
OUT = Path("src/data")
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


## 1. Carga de datos

In [2]:
orders = pd.read_csv(
    DATASET / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
items = pd.read_csv(DATASET / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATASET / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(
    DATASET / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
customers = pd.read_csv(DATASET / "olist_customers_dataset.csv")
products = pd.read_csv(DATASET / "olist_products_dataset.csv")
sellers = pd.read_csv(DATASET / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATASET / "product_category_name_translation.csv")
# El encabezado de la traducción trae BOM
category_translation.columns = [c.lstrip("\ufeff") for c in category_translation.columns]

print(f"orders: {len(orders):,} | items: {len(items):,} | payments: {len(payments):,}")
print(f"reviews: {len(reviews):,} | customers: {len(customers):,} | products: {len(products):,}")


orders: 99,441 | items: 112,650 | payments: 103,886
reviews: 99,224 | customers: 99,441 | products: 32,951


## 2. Preparación

- Una reseña por orden (la más reciente, si hay varias).
- Pago total por orden.
- Región geográfica del cliente a partir de su estado (UF).


In [3]:
# Una resena por orden: conservamos la mas reciente
reviews_per_order = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
)

# Pago total por orden (una orden puede tener varios registros de pago)
payment_per_order = payments.groupby("order_id", as_index=False)["payment_value"].sum()

# Region de Brasil por estado del cliente
STATE_TO_REGION = {
    **dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
    **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
    **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
    **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
    **dict.fromkeys(["PR", "RS", "SC"], "Sul"),
}
customers["region"] = customers["customer_state"].map(STATE_TO_REGION)

orders_full = (
    orders.merge(payment_per_order, on="order_id", how="left")
    .merge(reviews_per_order, on="order_id", how="left")
    .merge(customers[["customer_id", "customer_state", "region"]], on="customer_id", how="left")
)
orders_full["order_month"] = orders_full["order_purchase_timestamp"].dt.strftime("%Y-%m")

delivered = orders_full[orders_full["order_status"] == "delivered"].copy()
delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days

print(f"Ordenes totales: {len(orders_full):,} | entregadas: {len(delivered):,}")


Ordenes totales: 99,441 | entregadas: 96,478


## 3. KPIs generales → `kpis.json`

In [4]:
total_revenue = float(payments["payment_value"].sum())
total_orders = int(orders["order_id"].nunique())

# Ticket promedio: valor de la orden (productos + flete) promediado sobre ordenes con items
order_value = items.groupby("order_id").agg(total=("price", "sum"), freight=("freight_value", "sum"))
avg_ticket = float((order_value["total"] + order_value["freight"]).mean())

delivery_rate = float((orders["order_status"] == "delivered").mean() * 100)
avg_review = float(reviews_per_order["review_score"].mean())

late_mask = delivered["delay_days"] > 0
late_rate = float(late_mask.mean() * 100)
on_time_score = float(delivered.loc[~late_mask, "review_score"].mean())
late_score = float(delivered.loc[late_mask, "review_score"].mean())
late_revenue = float(delivered.loc[late_mask, "payment_value"].sum())

kpis = {
    "totalRevenue": round(total_revenue, 2),
    "totalOrders": total_orders,
    "avgTicket": round(avg_ticket, 2),
    "deliveryRate": round(delivery_rate, 2),
    "avgReviewScore": round(avg_review, 2),
    "totalCustomers": int(customers["customer_unique_id"].nunique()),
    "totalSellers": int(sellers["seller_id"].nunique()),
    "lateDeliveryRate": round(late_rate, 2),
    "onTimeAvgScore": round(on_time_score, 2),
    "lateAvgScore": round(late_score, 2),
    "lateOrdersRevenue": round(late_revenue, 2),
}
(OUT / "kpis.json").write_text(json.dumps(kpis, indent=2, ensure_ascii=False))
kpis


{'totalRevenue': 16008872.12,
 'totalOrders': 99441,
 'avgTicket': 160.58,
 'deliveryRate': 97.02,
 'avgReviewScore': 4.09,
 'totalCustomers': 96096,
 'totalSellers': 3095,
 'lateDeliveryRate': 6.77,
 'onTimeAvgScore': 4.29,
 'lateAvgScore': 2.27,
 'lateOrdersRevenue': 1150865.66}

## 4. Ingresos mensuales → `monthly_revenue.json`

In [5]:
monthly = (
    orders_full.dropna(subset=["payment_value"])
    .groupby("order_month", as_index=False)
    .agg(revenue=("payment_value", "sum"), orders=("order_id", "nunique"))
    .sort_values("order_month")
)
monthly["revenue"] = monthly["revenue"].round(2)

peak = monthly.loc[monthly["revenue"].idxmax()]
print(f"Pico historico: {peak['order_month']} con {peak['revenue']:,.2f} BRL")

monthly_records = monthly.to_dict(orient="records")
(OUT / "monthly_revenue.json").write_text(json.dumps(monthly_records, indent=2))
monthly.tail(8)


Pico historico: 2017-11 con 1,194,882.80 BRL


,order_month,revenue,orders
17,2018-03,"1,159,652.12",7211
18,2018-04,"1,160,785.48",6939
19,2018-05,"1,153,982.15",6873
20,2018-06,"1,023,880.50",6167
21,2018-07,"1,066,540.75",6292
22,2018-08,"1,022,425.32",6512
23,2018-09,"4,439.54",16
24,2018-10,589.67,4


## 5. Latencia de entrega vs. reseñas → `latency_review.json`

Agrupamos las órdenes entregadas por días de atraso (positivo) o anticipación
(negativo) respecto a la fecha estimada, y por región del cliente. Cada punto
lleva el score promedio y el número de órdenes.


In [6]:
lat = delivered.dropna(subset=["delay_days", "review_score", "region"]).copy()
lat["delay_days"] = lat["delay_days"].clip(-30, 30).astype(int)

def aggregate_latency(frame, region_name):
    grouped = (
        frame.groupby("delay_days", as_index=False)
        .agg(avgScore=("review_score", "mean"), orders=("order_id", "nunique"))
    )
    grouped["avgScore"] = grouped["avgScore"].round(2)
    grouped["region"] = region_name
    return grouped

frames = [aggregate_latency(lat, "Todas")]
for region_name in ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]:
    frames.append(aggregate_latency(lat[lat["region"] == region_name], region_name))

latency = pd.concat(frames, ignore_index=True).rename(columns={"delay_days": "delay"})

latency_payload = {
    "regions": ["Todas", "Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"],
    "points": latency[["region", "delay", "avgScore", "orders"]].to_dict(orient="records"),
}
(OUT / "latency_review.json").write_text(json.dumps(latency_payload, indent=2))
print(f"{len(latency_payload['points'])} puntos agregados")
latency[latency["region"] == "Todas"].query("delay in [-10, -5, 0, 5, 10, 20]")


362 puntos agregados


,delay,avgScore,orders,region
20,-10,4.33,4618,Todas
25,-5,4.19,2208,Todas
30,0,4.03,1280,Todas
35,5,2.20,434,Todas
40,10,1.58,207,Todas
50,20,1.63,86,Todas


## 6. Valor de pagos por score de reseña → `payment_by_score.json`

In [7]:
pay_score = (
    delivered.dropna(subset=["review_score", "payment_value"])
    .assign(review_score=lambda d: d["review_score"].astype(int))
    .groupby("review_score", as_index=False)
    .agg(paymentValue=("payment_value", "sum"), orders=("order_id", "nunique"))
    .rename(columns={"review_score": "score"})
    .sort_values("score")
)
pay_score["paymentValue"] = pay_score["paymentValue"].round(2)
pay_score["pctValue"] = (pay_score["paymentValue"] / pay_score["paymentValue"].sum() * 100).round(2)

(OUT / "payment_by_score.json").write_text(json.dumps(pay_score.to_dict(orient="records"), indent=2))
at_risk = pay_score.loc[pay_score["score"] <= 2, "paymentValue"].sum()
print(f"Revenue asociado a scores 1-2: {at_risk:,.2f} BRL")
pay_score


Revenue asociado a scores 1-2: 2,303,091.56 BRL


,score,paymentValue,orders,pctValue
0,1,"1,806,523.28",9352,11.81
1,2,"496,568.28",2920,3.25
2,3,"1,196,402.78",7917,7.82
3,4,"2,921,915.02",18893,19.10
4,5,"8,872,621.73",56749,58.01


## 7. Matriz comercial: categoría × volumen × flete × reseña → `category_matrix.json`

In [8]:
items_cat = (
    items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(reviews_per_order, on="order_id", how="left")
)

category = (
    items_cat.dropna(subset=["product_category_name"])
    .groupby(["product_category_name", "product_category_name_english"], as_index=False)
    .agg(
        volume=("order_id", "count"),
        revenue=("price", "sum"),
        avgFreight=("freight_value", "mean"),
        avgPrice=("price", "mean"),
        avgReview=("review_score", "mean"),
    )
    .rename(columns={
        "product_category_name": "category",
        "product_category_name_english": "categoryEn",
    })
    .sort_values("volume", ascending=False)
    .head(20)
)
for col in ["revenue", "avgFreight", "avgPrice", "avgReview"]:
    category[col] = category[col].round(2)

(OUT / "category_matrix.json").write_text(
    json.dumps(category.to_dict(orient="records"), indent=2, ensure_ascii=False)
)
category.head(10)


,category,categoryEn,volume,revenue,avgFreight,avgPrice,avgReview
13,cama_mesa_banho,bed_bath_table,11115,"1,036,988.68",18.42,93.30,3.90
11,beleza_saude,health_beauty,9670,"1,258,681.34",18.88,130.16,4.14
32,esporte_lazer,sports_leisure,8641,"988,048.97",19.51,114.34,4.11
54,moveis_decoracao,furniture_decor,8334,"729,762.49",20.73,87.56,3.90
44,informatica_acessorios,computers_accessories,7827,"911,954.32",18.82,116.51,3.93
70,utilidades_domesticas,housewares,6964,"632,248.66",20.99,90.79,4.05
64,relogios_presentes,watches_gifts,5991,"1,205,005.68",16.78,201.14,4.02
68,telefonia,telephony,4545,"323,667.53",15.67,71.21,3.95
40,ferramentas_jardim,garden_tools,4347,"485,256.46",22.77,111.63,4.05
8,automotivo,auto,4235,"592,720.11",21.88,139.96,4.06


## 8. Validación contra las cifras de referencia del plan

> **Nota sobre los ingresos totales:** el plan referencia 20,579,664.01 BRL, pero la
> suma real de `payment_value` es **16,008,872.12 BRL**. La cifra del plan proviene de
> un *double-count*: al hacer join de pagos con `order_items` (una fila por artículo),
> las órdenes multi-artículo duplican su pago (el join reproduce ≈20.3M). La
> presentación usa la cifra correcta calculada aquí.


In [9]:
reference = {
    # El plan decia 20,579,664.01 pero es un double-count (ver nota arriba)
    "totalRevenue": 16_008_872.12,
    "totalOrders": 99_441,
    "avgTicket": 160.99,
}
for key, expected in reference.items():
    actual = kpis[key]
    diff_pct = abs(actual - expected) / expected * 100
    status = "OK" if diff_pct < 1 else "REVISAR"
    print(f"{key}: actual={actual:,.2f} vs plan={expected:,.2f} ({diff_pct:.2f}% dif) [{status}]")

print()
print(f"Tasa de entrega: {kpis['deliveryRate']}% (plan: 97.13%)")
print(f"Score promedio: {kpis['avgReviewScore']} (plan: 4.02)")
print()
print("Archivos generados en src/data/:")
for f in sorted(OUT.glob("*.json")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")


totalRevenue: actual=16,008,872.12 vs plan=16,008,872.12 (0.00% dif) [OK]
totalOrders: actual=99,441.00 vs plan=99,441.00 (0.00% dif) [OK]
avgTicket: actual=160.58 vs plan=160.99 (0.25% dif) [OK]

Tasa de entrega: 97.02% (plan: 97.13%)
Score promedio: 4.09 (plan: 4.02)

Archivos generados en src/data/:
  category_matrix.json (3,862 bytes)
  kpis.json (295 bytes)
  latency_review.json (37,097 bytes)
  monthly_revenue.json (2,081 bytes)
  payment_by_score.json (495 bytes)
